In [1]:
from pathlib import Path

import rasterio as rio
import numpy as np

import geopandas as gpd
import pandas as pd

from tqdm import tqdm

In [2]:
wkdir = Path(r"G:\My Drive")



In [3]:
out_folder = Path(wkdir, 'GEE_GEDI_csvs')
out_folder.mkdir(exist_ok=True)


In [4]:
folder_list = list(wkdir.glob("GEE_exports_id*"))
print(len(folder_list))

162


In [75]:
folder_list = list(wkdir.glob("GEE_exports_id*"))
for folder in tqdm(folder_list):

    # Load the images
    img_list = list(folder.glob("*.tif"))
    
    for img_file in img_list:

        with rio.open(img_file) as src:
            img = src.read(1)
            img_meta = src.meta
            
        # Get the coordinates for each pixel
        x, y = np.meshgrid(np.arange(img_meta["width"]), np.arange(img_meta["height"]))
        x = src.xy(y.flatten(), x.flatten())[0].reshape(img.shape)
        y = src.xy(y.flatten(), x.flatten())[1].reshape(img.shape)
        # Write it to a dataframe with the pixel values
        df = pd.DataFrame({"x": x.flatten(), "y": y.flatten(), "agbd": img.flatten()})
        df['date'] = img_file.stem.split("_")[0]

        df.dropna(inplace=True, ignore_index=True, subset=["agbd"])

        if len(df) > 0:
            csv_out = Path(out_folder, f"{folder.stem}.csv")
            df.to_csv(csv_out, index=False, mode='a', header=False)
        else:
            continue
            #print(f"No data in {img_file.stem}")

array([nan])